# 05 — Ablations: which fields, which interactions

Requires  +  (produced by notebooks 02/03).

Compares content-field sets and a rating>=4 positive filter on NDCG@10 via the shared eval harness. Findings feed the model report.

In [1]:
import pandas as pd
from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.filters import filter_min_rating
from book_recsys.eval.harness import build_user_histories, build_relevance
from book_recsys.eval.experiment import run_experiments, results_table
from book_recsys.features.document import build_documents
from book_recsys.features.vectorize import tfidf_matrix
from book_recsys.models.content.content import ContentRecommender
from book_recsys.models.classical.svd import SvdRecommender


In [4]:
sample = pd.read_parquet("../artifacts/sample.parquet")
catalog = pd.read_parquet("../artifacts/catalog.parquet")
book_ids = catalog["book_id"].tolist()
train, holdout = leave_last_n_out(sample, n=1)
histories = build_user_histories(train)
relevance = build_relevance(holdout)


In [5]:
def content_rec(fields):
    matrix, _ = tfidf_matrix(build_documents(catalog, fields=fields))
    return ContentRecommender(book_ids, matrix).fit()

field_configs = {
    "title": content_rec(("title",)),
    "title+plot": content_rec(("title", "plot")),
    "title+plot+shelves": content_rec(("title", "plot", "shelves")),
}
results_table(run_experiments(field_configs, histories, relevance, k=10))


,recall@10,ndcg@10,mrr
title,0.00052,0.000250,0.000170
title+plot,0.00102,0.000484,0.000326
title+plot+shelves,0.00088,0.000450,0.000321


## TF-IDF vs BoW (full fields)

In [ ]:
from book_recsys.features.vectorize import bow_matrix
from book_recsys.models.content.content import ContentRecommender
tfidf_full, _ = tfidf_matrix(build_documents(catalog))
bow_full, _ = bow_matrix(build_documents(catalog))
rep_configs = {
    'content_tfidf_full': ContentRecommender(book_ids, tfidf_full).fit(),
    'content_bow_full': ContentRecommender(book_ids, bow_full).fit(),
}
results_table(run_experiments(rep_configs, histories, relevance, k=10))

In [ ]:
train_pos = filter_min_rating(train, min_rating=4)   # explicit likes only
configs = {
    "svd_all": SvdRecommender(n_factors=64).fit(train),
    "svd_rating>=4": SvdRecommender(n_factors=64).fit(train_pos),
}
results_table(run_experiments(configs, histories, relevance, k=10))


## Findings

<!-- Fill in after running the notebook against real artifacts. -->

- **Content ablation:** record which field set (k\, , or ) achieved the highest NDCG@10.
- **Interaction ablation:** record whether  or  won on NDCG@10.
- These rows feed the model report (Plan 5).
